# Zhang Figure 1 — recovered logs and replotting

This notebook **only plots saved, actual PyTorch training logs**. It does not retrain any model. It restores logs uploaded to ChatGPT by the user, preserving their original values. The MLP full-run `p=0.5` and `p=1` final logs were not among those backups. If recovered from the original Colab runtime, upload them and the plots will fill in automatically.

**Censored AlexNet `p=1`**: did not reach 99% training accuracy by 200 epochs; the x-marker at ratio 12.5 in panel (b) is a lower bound, and the 89.6% test error in panel (c) is from the final epoch, not convergence.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

print('Upload Zhang_Recovered_Logs.zip from this chat. Optionally also select final MLP JSON files.')
uploads = files.upload()
root = Path('Zhang_Recovered_Logs')
root.mkdir(exist_ok=True)
for fname, payload in uploads.items():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname) as z:
            z.extractall('.')
    elif fname.endswith('.json'):
        (root / fname).write_bytes(payload)
print('Restored',len(list(root.glob('*.json'))),'logs in',root.resolve())


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('Zhang_Recovered_Logs')
if not ROOT.exists():
    ROOT = Path('.')


def read(model, p, mode='true'):
    path = ROOT / f'{model}_p{p:.2f}_{mode}_n50000_reduced.json'
    if not path.exists():
        return None
    with path.open() as f:
        return json.load(f)

labels = {
    'true labels': ('inception', 0.0, 'true'),
    'random labels': ('inception', 1.0, 'true'),
    'shuffled pixels': ('inception', 0.0, 'shuffled_pixels'),
    'random pixels': ('inception', 0.0, 'random_pixels'),
    'Gaussian noise': ('inception', 0.0, 'gaussian'),
}

fig, (a,b,c) = plt.subplots(1, 3, figsize=(20, 6.7))
for label, (model, p, mode) in labels.items():
    r = read(model,p,mode)
    if r:
        l = r['batch_losses']
        a.plot([q['step']/1000 for q in l], [q['loss'] for q in l], label=label, linewidth=1.4)
a.set(xlabel='Thousand optimizer steps', ylabel='Minibatch cross-entropy',title='(a) Learning curves')
a.legend(fontsize=8)

for model in ('inception','alexnet','mlp1'):
    baseline = read(model,0)
    if not baseline or baseline.get('convergence_steps') is None:
        continue
    denom=baseline['convergence_steps']
    good_x,good_y=[],[]
    for p in (0,0.5,1.0):
        r=read(model,p)
        if r is None: continue
        if r.get('converged') and r.get('convergence_steps') is not None:
            good_x.append(p)
            good_y.append(r['convergence_steps']/denom)
    line,=b.plot(good_x,good_y,'o-',label=model)
    # Censored points: show LOWER BOUND at the elapsed budget, never pass off as convergence time
    for p in (0,0.5,1.0):
        r=read(model,p)
        if r is None or r.get('converged') or not r.get('completed'):continue
        lower_bound = r['training_steps']/denom
        b.plot(p,lower_bound,marker='x',markersize=10,linestyle='None',color=line.get_color())
        b.annotate(f'>{lower_bound:.1f}x (not converged)',(p,lower_bound), xytext=(-8,12),
                   textcoords='offset points',ha='right',color=line.get_color(),fontsize=8)
b.set(xlabel='Label replacement probability p',ylabel='Steps / true-label baseline',title='(b) Relative time to 99% train accuracy', xticks=[0,0.5,1])
b.legend(fontsize=9)

for model in ('inception','alexnet','mlp1'):
    gx,gy=[],[]
    for p in (0,0.5,1.0):
        r=read(model,p)
        if r and r.get('converged') and r.get('test_at_convergence'):
            gx.append(p); gy.append(1-r['test_at_convergence']['accuracy'])
    line,=c.plot(gx,gy,'o-',label=model)
    for p in (0,0.5,1.0):
        r=read(model,p)
        if r and r.get('completed') and not r.get('converged') and r.get('test_at_end'):
            c.plot(p,1-r['test_at_end']['accuracy'],marker='x',markersize=10,
                   linestyle='None',color=line.get_color())
            c.annotate('endpoint only',(p,1-r['test_at_end']['accuracy']),
                       xytext=(-8,-20),textcoords='offset points',ha='right',fontsize=8,color=line.get_color())
c.axhline(0.9,ls='--',color='grey',alpha=.7,label='90% chance error')
c.set(xlabel='Label replacement probability p',ylabel='Test classification error',title='(c) Test error (convergence/endpoint)',ylim=(0,1),xticks=[0,0.5,1])
c.legend(fontsize=9)
fig.suptitle('Recovered Zhang et al. Figure 1 experiments — actual CIFAR-10 training logs',fontsize=13)
fig.text(0.5,0.005,'X = training budget exhausted without convergence; panel (b) shows a lower bound. Missing MLP points are not imputed.',
         ha='center',fontsize=9)
fig.tight_layout(rect=(0,0.055,1,.94))
output=ROOT/'figure1_recovered_partial_with_alexnet_censored.png'
fig.savefig(output,dpi=180,bbox_inches='tight')
plt.show()
print('Saved',output)
print('Note: MLP p=0.5 final and p=1 final logs were not in the ChatGPT upload backups. Their points will appear automatically if those exact files are supplied in ROOT.')


In [ ]:
# Optional: persist recovered logs and plots to Drive so they survive runtime deletion.
# Run only if you want a Drive backup.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil
source = Path('Zhang_Recovered_Logs')
dest = Path('/content/drive/MyDrive/Zhang_Recovered_Logs')
shutil.copytree(source, dest, dirs_exist_ok=True)
print('Backup saved to:',dest)
